# Extended Grid — Colab GPU runtime for the heavy experiments

**Repository:** `the1finix/portfolio-risk-drl` (EEEM004 dissertation, Fiyinfoluwa Akano).

**What this notebook does, end-to-end, in one click:**

1. Clones the repo into the Colab VM and installs every dependency (~90 s on a fresh T4).
2. Smoke-tests CUDA + PyTorch + Stable-Baselines3 + the project's own modules.
3. Runs the **full 70-ticker × 10-seed × 50 000-PPO-step extended grid** with 32 stationary block-bootstrap training paths (the heaviest single experiment).
4. Runs the **eight-ticker × four-fold × 10-seed walk-forward grid at the extended timestep budget** (the strongest out-of-sample evidence).
5. Optionally runs the **70-ticker walk-forward grid** (very heavy — only enable on A100 / V100).
6. Aggregates results and rebuilds **`Main_Dissertation_Draft.docx`** and **`Fiyins_Dissertation.docx`** with the new numbers folded in.
7. Zips everything (results + charts + docs) and offers a one-click download / Drive mount.

**Why this is on GPU and not CPU.** The probabilistic PPO agent dominates total wall-clock. A 70-ticker × 10-seed × 50 000-step run is ~2–3 days on an M-series MacBook CPU; on a Colab T4 it is ~5–7 hours (~2 hours on A100). The walk-forward grid scales with `tickers × folds × seeds × timesteps`, so a 4-ticker × 4-fold × 10-seed × 50k-step grid that takes 3+ hours on CPU is ~20 minutes on T4.

**Runtime preset.** *Runtime → Change runtime type → Hardware accelerator: T4 GPU* (A100 if you have Pro+). Then *Runtime → Run all*.

> Heavy work belongs here. The CPU pipeline (`experiments/run_extended_grid.py --tickers basket --skip-walk-forward`) is the smoke-test path; this notebook is the canonical home for everything bigger.

## 1 · Setup — clone, install, GPU sanity

Run the next cell once per Colab session. It is idempotent: if the repo is already cloned, `git pull` brings it up to date; if the dependencies are already installed, `pip install` is a no-op.

In [ ]:
import os, subprocess, sys, shlex, time, pathlib, json

REPO_URL  = "https://github.com/the1finix/portfolio-risk-drl.git"
REPO_NAME = "portfolio-risk-drl"
WORKDIR   = pathlib.Path("/content") / REPO_NAME

def sh(cmd, cwd=None, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, cwd=cwd, shell=True, text=True, capture_output=False)
    if check and r.returncode != 0:
        raise SystemExit(f"command failed: {cmd}")
    return r.returncode

if not WORKDIR.exists():
    sh(f"git clone --depth 1 {REPO_URL} {WORKDIR}")
else:
    sh("git pull --ff-only", cwd=str(WORKDIR), check=False)

os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
sh("pip install -q -r requirements.txt")

import torch
print(f"\nPython:        {sys.version.split()[0]}")
print(f"PyTorch:       {torch.__version__}")
print(f"CUDA built:    {torch.version.cuda}")
print(f"CUDA available:{torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device:        {torch.cuda.get_device_name(0)}")
    print(f"Mem (GiB):     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")
else:
    print("WARNING — no GPU detected. Switch the runtime to T4 / A100 or this notebook will be useless.")


## 2 · GPU smoke test — one ticker, one seed, 5 000 timesteps

Verifies that the probabilistic agent actually moves data through the GPU before launching anything heavy. Should finish in ~1 minute on a T4.

In [ ]:
t0 = time.time()
sh(
    "python experiments/run_probabilistic_agent.py "
    "--tickers SPY --seeds 7 --timesteps 5000 --tag _gpu_smoke"
)
print(f"\nGPU smoke test finished in {time.time() - t0:.0f} s.")


## 3 · The heaviest run — 70-ticker × 10-seed × 50 000-step extended grid (with bootstrap)

This is the experiment that matters. Single command:

* 70 tickers (every holding in `fiyins_portfolio`)
* 10 random seeds per ticker
* 50 000 PPO timesteps per (ticker, seed) cell
* 32 stationary block-bootstrap (Politis & Romano 1994) augmentation paths per training run
* All four agents — buy-and-hold, manual 5 % stop-loss, baseline PPO, probabilistic PPO

Walk-forward is **skipped** in this cell so the run finishes within a single T4 session (~5–7 hours).
The dedicated walk-forward grid is the next cell.

Output files land in `experiments/results/*_colab70.json` and overwrite nothing else.

In [ ]:
t0 = time.time()
sh(
    "python experiments/run_extended_grid.py "
    "--tickers fiyins_portfolio "
    "--seeds extended "
    "--timesteps 50000 "
    "--bootstrap-paths 32 "
    "--tag colab70 "
    "--skip-walk-forward"
)
elapsed = time.time() - t0
print(f"\n70-ticker extended grid finished in {elapsed/3600:.2f} h "
      f"({elapsed/60:.1f} min, {elapsed:.0f} s).")


## 4 · Walk-forward at extended budget — 8-ticker basket × 4 folds × 10 seeds × 50 000 steps

This is the strongest out-of-time evidence. 320 individual PPO trainings (8 tickers × 4 folds × 10 seeds, with both baseline and probabilistic agents) on the four protocol folds wf_2018_2019, wf_2020_2021, wf_2022_2023 and wf_2024_2025. Each fold trains on the strictly earlier window and evaluates on the strictly later window — the agent never sees the test data during training.

Expected runtime on T4: ~3–4 hours. On A100: ~1–1.5 hours.

In [ ]:
t0 = time.time()
sh(
    "python experiments/run_walk_forward.py "
    "--tickers basket "
    "--folds all "
    "--seeds extended "
    "--timesteps 50000 "
    "--bootstrap-paths 16 "
    "--agents baseline,probabilistic "
    "--tag colab_wf_basket"
)
elapsed = time.time() - t0
print(f"\nBasket walk-forward grid finished in {elapsed/3600:.2f} h "
      f"({elapsed/60:.1f} min, {elapsed:.0f} s).")


## 5 · *(Optional, A100 only)* — full 70-ticker walk-forward

The most extensive single experiment in the project: 70 tickers × 4 folds × 10 seeds × 50 000 timesteps × 16 bootstrap paths × two agents = ~5 600 individual trainings. Do **not** run on a T4 — the session will time out. Enable only on A100 / V100, where it takes ~12–14 hours.

`RUN_FULL_70_WF = True` flips it on.

In [ ]:
RUN_FULL_70_WF = False  # flip to True only on A100 / V100

if RUN_FULL_70_WF:
    t0 = time.time()
    sh(
        "python experiments/run_walk_forward.py "
        "--tickers fiyins_portfolio "
        "--folds all "
        "--seeds extended "
        "--timesteps 50000 "
        "--bootstrap-paths 16 "
        "--agents baseline,probabilistic "
        "--tag colab_wf_fiyins70"
    )
    elapsed = time.time() - t0
    print(f"\nFull 70-ticker walk-forward finished in {elapsed/3600:.2f} h.")
else:
    print("Skipped — set RUN_FULL_70_WF = True on an A100/V100 runtime to enable.")


## 6 · Aggregate, rebuild documents, generate charts

Pulls the new result JSONs from `experiments/results/`, aggregates them, regenerates the 70-ticker case-study charts and tables, and rebuilds `Main_Dissertation_Draft.docx` + `Fiyins_Dissertation.docx` with the extended-budget numbers folded in.

In [ ]:
sh("python experiments/aggregate_results.py --tag colab70")
sh("python reports/build_fiyins_case_study.py")
sh("python reports/build_fiyins_dissertation_docx.py")
sh("python reports/build_main_dissertation_docx.py")
sh("python reports/build_interim_review_docx.py")

print("\nGenerated documents:")
for p in sorted(pathlib.Path("reports/generated/exports").glob("*.docx")):
    print(f"  {p.relative_to(WORKDIR)}  ({p.stat().st_size/1024:.0f} KiB)")
print("\nGenerated charts:")
for p in sorted(pathlib.Path("reports/generated/charts").glob("*.png")):
    print(f"  {p.relative_to(WORKDIR)}  ({p.stat().st_size/1024:.0f} KiB)")


## 7 · Quick at-a-glance summary of the new results

A small in-notebook print-out so you can verify the numbers landed before downloading. Reads the latest `*_colab70` result file and prints the four-agent comparison and a per-ticker drawdown-reduction count.

In [ ]:
import json, statistics as st
from pathlib import Path

results = Path("experiments/results")

def latest(prefix, tag):
    files = sorted(results.glob(f"{prefix}_*_{tag}.json"))
    return json.loads(files[-1].read_text()) if files else []

bench = latest("benchmarks", "colab70")
rule  = latest("rule_baseline", "colab70")
base  = latest("baseline", "colab70")
prob  = latest("probabilistic", "colab70")

def mean_metric(rows, key, agent_filter=None):
    if agent_filter:
        rows = [r for r in rows if r.get("agent") == agent_filter]
    vals = [float(r[key]) for r in rows if r.get(key) is not None]
    return sum(vals) / len(vals) if vals else float("nan")

bh   = [r for r in bench if r.get("agent") == "buy_and_hold"]
stop = [r for r in rule  if r.get("agent") == "stop_loss_5pct"]

print("=== 70-ticker extended grid (tag = colab70) — aggregate ===")
print(f"  Buy-and-hold       final={mean_metric(bh,   'final_portfolio_value'):>14,.0f}   sharpe={mean_metric(bh,   'sharpe_ratio'):+.2f}   mdd={mean_metric(bh,   'max_drawdown'):.3f}")
print(f"  Manual 5% stop     final={mean_metric(stop, 'final_portfolio_value'):>14,.0f}   sharpe={mean_metric(stop, 'sharpe_ratio'):+.2f}   mdd={mean_metric(stop, 'max_drawdown'):.3f}")
print(f"  Baseline PPO       final={mean_metric(base, 'final_portfolio_value'):>14,.0f}   sharpe={mean_metric(base, 'sharpe_ratio'):+.2f}")
print(f"  Probabilistic PPO  final={mean_metric(prob, 'final_portfolio_value'):>14,.0f}   sharpe={mean_metric(prob, 'sharpe_ratio'):+.2f}   mdd={mean_metric(prob, 'max_drawdown'):.3f}")

bh_by_t = {r["ticker"]: r for r in bh}
prob_by_t: dict[str, list[float]] = {}
for r in prob:
    prob_by_t.setdefault(r["ticker"], []).append(float(r["max_drawdown"]))
dd_wins = sum(
    1 for t, mdds in prob_by_t.items()
    if t in bh_by_t and st.median(mdds) < float(bh_by_t[t]["max_drawdown"])
)
print(f"\nDrawdown reduction vs B&H: {dd_wins} of {len(prob_by_t)} tickers")


## 8 · Package outputs — one zip, two delivery options

* **Option A · Direct download.** Smallest convenience: zip everything important and trigger a browser download.
* **Option B · Google Drive mount.** For multi-day grids on a Pro+ runtime: copy the zip to `/MyDrive/portfolio-risk-drl/` so the next session can resume from disk.

In [ ]:
import shutil, datetime

stamp = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
bundle_name = f"portfolio_risk_drl_extended_{stamp}"
bundle_dir = pathlib.Path("/content") / bundle_name
bundle_dir.mkdir(exist_ok=True)

for src in [
    "experiments/results",
    "reports/generated/exports",
    "reports/generated/charts",
    "reports/generated/figures",
]:
    p = pathlib.Path(src)
    if p.exists():
        shutil.copytree(p, bundle_dir / p.name, dirs_exist_ok=True)

archive = shutil.make_archive(str(bundle_dir), "zip", bundle_dir)
size_mb = pathlib.Path(archive).stat().st_size / 1e6
print(f"\nBundle: {archive}  ({size_mb:.1f} MiB)")

# --- Option A: direct download
try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print(f"(skipping browser download: {exc})")

# --- Option B: copy to Drive (uncomment to enable)
# from google.colab import drive
# drive.mount("/content/drive", force_remount=False)
# drive_target = pathlib.Path("/content/drive/MyDrive/portfolio-risk-drl")
# drive_target.mkdir(parents=True, exist_ok=True)
# shutil.copy2(archive, drive_target / pathlib.Path(archive).name)
# print(f"Copied to {drive_target / pathlib.Path(archive).name}")


---

## What this notebook does *not* do (and why)

Everything that the project's CPU pipeline already handles cheaply is intentionally **left to CPU** — the smoke-tests, the eight-ticker basket at the Phase-1 budget, the four-ticker × four-fold walk-forward grid at 10 000 timesteps, the document builds, the supervisor-pack zip. They do not benefit from a GPU and they would just consume Colab quota.

Conversely, the items below were **specifically pushed off CPU and into this notebook** because the CPU runtime makes them either painfully slow or outright infeasible:

| Experiment                                                    | CPU runtime           | Colab T4 runtime | Notebook cell |
| ---                                                           | ---                   | ---              | --- |
| 70-ticker × 10-seed × 50 000-step extended grid + bootstrap   | 2–3 days              | 5–7 hours        | Section 3 |
| 8-ticker × 4-fold × 10-seed × 50 000-step walk-forward        | 8–12 hours            | 3–4 hours        | Section 4 |
| Full 70-ticker walk-forward (A100 only)                       | infeasible            | 12–14 h on A100  | Section 5 |
| Document rebuild with extended numbers folded in              | (depends on grid)     | < 1 minute       | Section 6 |

If a future experiment is heavier than the eight-ticker basket at Phase-1 budget, add it here rather than to the CPU pipeline.